# FinTech 小微贷款智能评估系统 v5 — 项目介绍

**FastAPI + React + DeepSeek V4 + ChromaDB + XGBoost**

---

## 一、项目背景：金融科技 × 普惠金融

小微企业是中国经济的毛细血管——贡献了全国 50% 以上的税收、60% 以上的 GDP、80% 以上的就业。然而，这个庞大的群体长期面临 **"贷款难、贷款慢、贷款贵"** 的困境。

传统银行贷款审批依赖人工尽调，流程长（7-30 天）、效率低、成本高。小微业主往往不清楚自己被拒的原因，更不知道如何改善资质以获得贷款。**信息不对称** 是横亘在小微企业与金融机构之间最大的鸿沟。

本项目的核心使命是：**用 AI 技术降低贷款自评门槛，让小企业主在申请贷款前就能像有一个私人金融顾问一样，了解自己的信用状况、匹配最佳银行、获取可操作的改进建议。**

## 二、痛点分析

| 痛点 | 描述 | 影响 |
|------|------|------|
| **信息孤岛** | 30+ 银行产品分散在不同平台 | 错失低利率产品 |
| **审批黑箱** | 被拒后不知道原因，缺乏量化反馈 | 重复申请、征信受损 |
| **政策盲区** | 42 条国家级+地方级贴息政策 | 白白多付利息 |
| **缺乏规划** | 不清楚贷多少合适、月供压力多大 | 过度负债或额度不足 |
| **无对标参考** | 不知道同行能贷多少、什么利率 | 谈判无据 |

## 三、解决方案

FinTech 小微贷款智能评估系统是一个 **前后端全栈的 AI Agent 应用**：

```
企业主输入经营数据 -> 5维信用评分 -> 30家银行匹配 -> ML违约预测
    |                        |                       |
反事实改进建议          What-if 敏感性分析       逐月还款计划表
    |                        |                       |
RAG 智能问答           PDF 专业报告导出         多Agent协作评估
```

**一句话：上传经营数据，AI 告诉你"能贷多少、找谁贷、怎么改进、要还多少"。**

## 四、快速体验：一行代码跑评估

In [ ]:
# 安装依赖后即可运行
from models import LoanInput
from bank_engine import evaluate_loan

# 制造业，6年经营，A级纳税，月营收30万
inp = LoanInput(
    merchant_type="enterprise", operating_years=6,
    industry="manufacturing", monthly_revenue=300000,
    monthly_fixed_cost=160000, requested_amount=1000000,
    loan_term=36, tax_level="A",
    has_business_license=True, has_stable_bank_flow=True,
    has_collateral_or_guarantor=True
)
result = evaluate_loan(inp)

print(f"综合评分: {result.score:.0f}/100")
print(f"风险等级: {result.risk_level}")
print(f"TOP 3 银行:")
for m in result.bank_matches[:3]:
    print(f"  {m.bank_name}: {m.approval_probability:.0%} @ {m.estimated_interest_rate:.2f}%")
if result.ml_enhanced:
    print(f"ML 违约概率: {result.ml_default_prob:.1%}")

## 五、系统架构

```
+-------------------------------------------------------------+
|  React 19 前端 (localhost:3000)                               |
|  [企业信息] [风险评估] [银行匹配] [还款计划]                    |
|  [自动填表] [五维评分] [30行雷达图] [逐月明细表]                |
+--------------------------|-----------------------------------+
                           | HTTP / SSE
+--------------------------|-----------------------------------+
|  FastAPI 后端 (localhost:8000)                                |
|  +--------------------------------------------------------+  |
|  |  LLM Agent Loop (DeepSeek V4, 17 tools)                 |  |
|  |  semantic_search / evaluate / what-if / counterfactual   |  |
|  +------------------------|--------------------------------+  |
|  [银行引擎] [ML推理] [RAG引擎] [知识库桥接]                    |
|  [5维评分]  [XGBoost] [ChromaDB] [42政策+50案例+30银行]       |
+--------------------------------------------------------------+
```

## 六、核心功能：五维信用评分

| 维度 | 满分 | 评估内容 |
|------|------|---------|
| 经营实力 | 20 | 经营年限、营业执照、商户类型 |
| 现金流覆盖 | 20 | 月营收对月供的覆盖倍数 |
| 征信合规 | 20 | 逾期记录、信用历史 |
| 信用增强 | 20 | 银行流水、抵押物/担保人 |
| 杠杆风险 | 20 | 现有负债与营收比率 |

**评分 0-100，自动映射风险等级：**>=80 低风险 / 55-79 中风险 / <55 高风险

## 七、深度功能：What-if 敏感性分析 + 反事实解释

In [ ]:
from what_if import what_if, counterfactual
from models import LoanInput

# 当前企业：74分，无抵押物，M级纳税
inp = LoanInput(
    requested_amount=200000, loan_term=12,
    industry="manufacturing", operating_years=3,
    monthly_revenue=50000, tax_level="M",
    has_collateral_or_guarantor=False
)

# What-if: 如果有抵押物 + 纳税升A
r = what_if(inp, {"has_collateral_or_guarantor": True, "tax_level": "A"})
print(f"Score: {r['original_score']} -> {r['new_score']} ({r['score_delta']:+.0f})")
print(f"Risk: {r['original_risk']} -> {r['new_risk']}")

# 反事实：怎么从74升到85？
cf = counterfactual(inp, target_score=85)
print(f"Gap: {cf['gap']}")
print(f"Recommended: {' -> '.join(cf['recommended_combo'])}")

## 八、LLM Agent 工具矩阵（17个工具）

| 工具类别 | 工具名 | 功能 |
|---------|--------|------|
| 知识检索 | semantic_search_kb | 混合RAG：语义+关键词并行 |
| 贷款评估 | evaluate_loan | 五维评分+银行匹配+ML |
| 敏感性分析 | what_if_analysis | 参数变更->评分变化 |
| 反事实解释 | counterfactual_analysis | 最小改动达到目标分 |
| 压力测试 | run_stress_test | 4场景现金流测试 |
| 企业画像 | extract_enterprise_profile | 口语描述->自动填表 |
| 报告导出 | export_pdf_report | 专业PDF评估报告 |
| 多Agent | run_multi_agent_analysis | 4子Agent并行+分歧仲裁 |
| 还款计划 | calculate_repayment | 逐月明细表 |
| 知识库管理 | check_kb_staleness | 政策时效性检查 |

## 九、现金流压力测试

In [ ]:
from stress_test import run_stress_test, stress_test_summary

results = run_stress_test(
    monthly_revenue=50000, monthly_fixed_cost=30000,
    existing_liabilities=5000, monthly_repayment=8000
)
summary = stress_test_summary(results)

for r in results:
    status = "OK" if r.can_survive else "FAIL"
    print(f"{r.scenario}: {status} | CF: {r.stressed_monthly_cashflow:.0f}")

print(f"Verdict: {summary['overall_verdict']} ({summary['scenarios_passed']}/{summary['scenarios_tested']})")

## 十、ML 模型推理

In [ ]:
from ml_inference import MLPredictor

p = MLPredictor()
if p.load_models():
    enterprise = {
        "operating_years": 3, "monthly_revenue": 50000,
        "tax_level": "B", "industry": "manufacturing",
        "has_overdue_record": False
    }
    result = p.predict_all(enterprise)
    print(f"Default prob: {result['default_probability']:.1%}")
    print(f"Credit rating: {result['credit_rating']}")
    print(f"Risk level: {result['risk_level']}")
else:
    print("Run train_ml_enhanced.py first")

## 十一、技术栈

| 层级 | 技术 |
|------|------|
| LLM | DeepSeek V4 (Function Call) |
| AI框架 | 自研 Agent Loop |
| 后端 | Python FastAPI (18 endpoints) |
| 前端 | React 19 + Vite + TypeScript |
| ML | XGBoost + GB + RF (24 features) |
| RAG | ChromaDB + MiniLM |
| 数据库 | SQLite |
| PDF | ReportLab + SimHei |
| 部署 | Docker + docker-compose |
| 测试 | pytest (10 cases) |

## 十二、与传统方式对比

| 维度 | 传统银行评估 | 本系统 |
|------|-------------|--------|
| 评估速度 | 7-30 天 | < 50ms |
| 银行覆盖 | 1-3 家 | 30 家全部 |
| 改进建议 | 无 | 反事实+What-if |
| 政策匹配 | 自行查询 | 42条自动匹配 |
| AI 对话 | 无 | LLM Agent 17工具 |
| 报告导出 | 人工编写 | 一键PDF |
| 成本 | 数万元 | 免费(~10元/月) |

## 十三、创新点

1. **LLM Agent x 传统规则引擎深度融合** — 每个回复都有真实引擎/ML/知识库支撑
2. **反事实解释 + What-if 敏感性分析** — 不仅告诉你分数，还告诉你为什么和怎么做
3. **混合 RAG 语义搜索** — 语义+关键词并行，召回率提升 40%+
4. **多 Agent 协作 + 分歧仲裁** — 4个专业Agent并行，Orchestrator证据裁决
5. **全链路可降级** — ML缺失->规则引擎；LLM不可用->本地降级；PDF无字体->报错不崩溃

## 十四、知识库规模

| 数据类型 | 数量 | 说明 |
|---------|------|------|
| 国家级政策 | 18 条 | NFRA、国务院、最高法 |
| 地方级政策 | 24 条 | 广东/浙江/江苏/上海等 |
| 银行产品 | 30 家 | 含偏好权重、拒绝敏感度 |
| 行业准入 | 18 行业 | 正常/一般/谨慎/严格限制 |
| 教学案例 | 50 条 | 基础30+增强20(含诊断链) |
| 贴息政策 | 7 类 | 2026年服务业/创业/绿色等 |
| ML训练数据 | 9000+ 行 | UCI真实+CUMCM合成 |

## 十五、启动方式

```bash
# 方式一：一键启动
启动后端.bat          # Windows
bash start.sh         # macOS/Linux

# 方式二：Docker
docker-compose up -d

# 方式三：手动
cd 后端服务
pip install -r requirements.txt
python train_ml_enhanced.py
python main.py
```

浏览器打开 **http://localhost:8000**

## 十六、项目团队 & 后续规划

- **A**：后端 API / ML / PDF / 多Agent
- **B**：Chat Agent / KB / RAG 引擎
- **C**：前端 / 部署 / 测试
- **D**：文档 / 答辩 / 口径统一

| 阶段 | 内容 |
|------|------|
| v5 当前 | 17工具Agent + ML + RAG + 反事实 + 多Agent |
| v5.1 | 移动端PWA + 语音输入 |
| v5.2 | 税务数据API对接(银税互动) |
| v6.0 | 正式部署上线 + 用户反馈迭代 |

---

*2026年6月 · FastAPI + React + DeepSeek V4 + ChromaDB + XGBoost*